# Graficos Resumen y Matriz de Competicion
Lee **solo los resultados de la Parte 1 (competicion)** de los 16 cuadernos.
Requiere que cada cuaderno haya sido ejecutado y haya generado su JSON en `../resultados/metricas/`.


In [ ]:
import sys
sys.path.insert(0, '..')
import os
import pandas as pd
from utilidades.evaluacion import cargar_todos_resultados
from utilidades.graficos   import graficar_resumen, graficar_matriz_competicion


## Carga de resultados de competicion

In [ ]:
# seccion='competicion' filtra solo los modelos de la Parte 1
df = cargar_todos_resultados('../resultados/metricas/', seccion='competicion')

combinaciones = df.groupby(['ventana_entrada','ventana_salida']).ngroups
print(f'Combinaciones cargadas: {combinaciones} / 16')
if combinaciones < 16:
    faltantes = [(e, s) for e in [5,10,30,90] for s in [1,5,30,90]
                 if df[(df.ventana_entrada==e)&(df.ventana_salida==s)].empty]
    print('Faltan:', faltantes)

display(df.pivot_table(index='ventana_entrada', columns='ventana_salida',
                        values='mae_test', aggfunc='min').round(5))


## Graficos resumen por ventana de salida (Parte 1 — Competicion)

In [ ]:
for ventana_sal in [1, 5, 30, 90]:
    graficar_resumen(df, ventana_sal)


## Matriz de competicion — mejor MAE test por combinacion

In [ ]:
graficar_matriz_competicion(df)


## Efecto del preprocesado (Parte 2 — Investigacion)

In [ ]:
# Comparativa entre el mejor modelo sin y con preprocesado, por combinacion
df_inv = cargar_todos_resultados('../resultados/metricas/', seccion='investigacion')

if df_inv.empty:
    print('Sin datos de investigacion todavia. Ejecuta la Parte 2 de los cuadernos.')
else:
    # Para cada combinacion: mejor MAE test de competicion vs investigacion
    mejor_comp = df.groupby(['ventana_entrada','ventana_salida'])['mae_test'].min().rename('mae_competicion')
    mejor_inv  = df_inv.groupby(['ventana_entrada','ventana_salida'])['mae_test'].min().rename('mae_investigacion')
    comparativa = pd.concat([mejor_comp, mejor_inv], axis=1).round(6)
    comparativa['mejora_%'] = ((comparativa['mae_competicion'] - comparativa['mae_investigacion'])
                                / comparativa['mae_competicion'] * 100).round(1)
    display(comparativa)
